# SolarVision v2 — parcours de bout en bout

Prévision de la production photovoltaïque journalière (1 kWc) de J+1 à J+7, comparaison de trois technologies
de modules, bénéfices économiques.

Ce notebook déroule toutes les étapes sur **trois villes** (Paris, Tunis, Madrid) : c'est le mode rapide du pipeline.
L'étude complète se lance par `python -m solarvision.cli pipeline`. L'étude statistique détaillée de Tunis
(ACF et PACF, STL, ARIMA et SARIMA, VAR, Granger, Johansen, GARCH) est dans `mini_projet_python.ipynb` et dans le notebook R.

Mettre `FAST = False` pour travailler sur toutes les villes déjà traitées.

In [1]:
# Versions et graine : le notebook doit donner les mêmes résultats d'une exécution à l'autre.
import platform, random, warnings
from importlib import metadata

import numpy as np
import pandas as pd

from solarvision.config import DATA, REPORTS, load_cities, load_params, select_cities

warnings.filterwarnings("ignore")
FAST = True
params = load_params()
random.seed(params["seed"]); np.random.seed(params["seed"])
print("Python", platform.python_version(), "| graine", params["seed"])
for lib in ("numpy", "pandas", "statsmodels", "xgboost", "prophet", "torch", "scikit-learn", "pandera"):
    print(f"  {lib:14s} {metadata.version(lib)}")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

Python 3.14.2 | graine 42
  numpy          2.3.5
  pandas         2.3.3
  statsmodels    0.15.0
  xgboost        3.1.3
  prophet        1.5.0
  torch          2.12.1
  scikit-learn   1.8.0
  pandera        0.34.1


## 1. Données

Trois sources : PVGIS (production simulée de trois technologies), Open-Meteo (météo ERA5) et NASA POWER
(rayonnement de contrôle). Les réponses sont mises en cache ; rien n'est retéléchargé s'il est déjà présent.

In [2]:
# Téléchargement de ce qui manque, puis construction des tables journalières (imputation et contrôles compris).
from solarvision.data import ingest, process

cities = select_cities(fast=FAST)
root = DATA / "fast" if FAST else DATA
ingest.run_downloads(cities)
quality = process.process_all(cities, root)
quality[["ville", "lignes", "jours_imputes", "manquants_pct", "statut"]]

17:01:45 PVGIS : 0 appels horaires à faire pour 3 villes


17:01:45 Téléchargements terminés : 0 échec(s), détail dans data/failures.json


17:01:46 paris : table traitée, 11 jours imputés, statut ok


17:01:48 tunis : table traitée, 13 jours imputés, statut ok


17:01:49 madrid : table traitée, 13 jours imputés, statut ok


,ville,lignes,jours_imputes,manquants_pct,statut
0,paris,6939,11,0.0,ok
1,tunis,6939,13,0.0,ok
2,madrid,6939,13,0.0,ok


In [3]:
# Table traitée de Tunis : les trois séries de production et le drapeau d'imputation.
tunis = pd.read_csv(root / "processed" / "tunis.csv", parse_dates=["date"]).set_index("date")
print(tunis.describe().loc[["mean", "std", "min", "max"]].round(3))
print("\nJours dont le rayonnement PVGIS manquait (corrigés avec NASA POWER) :")
print([str(d.date()) for d in tunis.index[tunis["jour_impute_pvgis"] == 1]])

      pv_crystSi_kwh_kwc  pv_CIS_kwh_kwc  pv_CdTe_kwh_kwc  jour_impute_pvgis
mean               4.313           4.271            4.413              0.002
std                1.392           1.387            1.490              0.043
min                0.211           0.225            0.134              0.000
max                6.487           6.371            6.557              1.000

Jours dont le rayonnement PVGIS manquait (corrigés avec NASA POWER) :
['2005-04-18', '2008-01-23', '2008-03-01', '2008-03-06', '2008-05-17', '2008-05-18', '2009-04-18', '2009-04-19', '2009-08-15', '2009-08-16', '2014-12-02', '2016-10-16', '2016-10-17']


## 2. Panel et découpage

Une ligne par ville et par jour. Chaque colonne a un rôle : cible, variable **ex ante** (connue avant le jour visé)
ou variable **ex post** (météo observée le jour même). Le découpage est chronologique : 90 % des jours les plus
anciens pour l'apprentissage.

In [4]:
from solarvision.features import panel as pm

panel = pm.build_panel(root, allow_partial=True, cities=cities)
splits, roles = pm.load_splits(root), pm.load_roles(root)
print(f"Panel : {len(panel)} lignes, {panel.shape[1]} colonnes, {panel['ville'].nunique()} villes")
print("Découpage de Tunis :", splits["tunis"])
print({k: len(v) for k, v in roles.items()})

Panel : 20817 lignes, 61 colonnes, 3 villes
Découpage de Tunis : {'n': 6939, 'n_train': 6245, 'n_test': 694, 'train_debut': '2005-01-01', 'train_fin': '2022-02-05', 'test_debut': '2022-02-06', 'test_fin': '2023-12-31'}
{'identifiants': 8, 'cibles': 3, 'ex_ante': 28, 'ex_post': 21, 'drapeaux': 1}


## 3. Exploration

Statistiques sur l'apprentissage seulement. L'ADF a pour hypothèse nulle une racine unitaire ; le KPSS a pour
hypothèse nulle la stationnarité.

In [5]:
from solarvision.reporting import exploration

reports = REPORTS / "fast" if FAST else REPORTS
stats = exploration.run(panel, splits, reports)
stats[["ville", "moyenne", "variance", "skewness", "kurtosis_exces", "rapport_saisonnier", "acf1_ecart_normale",
       "adf_p_ecart_normale", "kpss_p_ecart_normale", "stationnaire_ecart_normale"]].round(3)

,ville,moyenne,variance,skewness,kurtosis_exces,rapport_saisonnier,acf1_ecart_normale,adf_p_ecart_normale,kpss_p_ecart_normale,stationnaire_ecart_normale
0,madrid,4.422,2.259,-1.062,0.084,1.612,0.413,0.0,0.100,True
1,paris,3.024,3.350,0.158,-1.173,3.092,0.362,0.0,0.050,False
2,tunis,4.295,1.965,-0.815,-0.278,1.771,0.258,0.0,0.079,True


In [6]:
# La série de Tunis et sa normale saisonnière (calculée sur l'apprentissage, lissée sur 31 jours).
import matplotlib.pyplot as plt
from solarvision.reporting import figures  # applique le style commun des figures

t = panel[panel["ville"] == "tunis"].set_index("date")
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t.index, t["pv_crystSi_kwh_kwc"], color=figures.MUTED, linewidth=0.4, label="Production journalière")
ax.plot(t.index, t["normale_crystSi"], color=figures.SERIES[0], linewidth=2, label="Normale saisonnière")
ax.axvline(pd.Timestamp(splits["tunis"]["test_debut"]), color=figures.AXIS, linestyle="--")
ax.set_ylabel("kWh/kWc par jour"); ax.set_title("Tunis : production et normale saisonnière"); ax.legend(loc="lower left", ncol=2)
plt.show()

## 4. Modèles classiques

Deux références (persistance, climatologie), une régression harmonique à erreurs AR(1), XGBoost en prévision directe
et Prophet. Depuis **chaque** jour du test, chaque modèle prévoit J+1 à J+7 ; les paramètres sont réestimés tous les 30 jours.

In [7]:
# Le modèle statistique à la main, sur Tunis : estimation, puis prévision depuis chaque origine.
from solarvision.evaluation.metrics import compute_metrics
from solarvision.models import baselines, statistical
from solarvision.models.protocol import origins, target_matrix
from solarvision.models.train import FOURIER, load_city

series = load_city("tunis", root)
y, exog = series.frame["pv_crystSi_kwh_kwc"].to_numpy(), series.frame[FOURIER].to_numpy()
pos = origins(series.n, series.n_train)
fit = statistical.fit_ar1_regression(y[: series.n_train], exog[: series.n_train])
print(f"AR(1) : phi = {fit['phi']:.3f}, écart-type des innovations = {fit['sigma2'] ** 0.5:.3f} kWh/kWc, BIC = {fit['bic']:.0f}")
observed = target_matrix(y, pos)
candidates = {"persistance": baselines.persistence(y, pos),
              "climatologie": baselines.climatology(series.frame["normale_crystSi"].to_numpy(), pos),
              "arima_fourier": statistical.forecast_ar1_regression(y, exog, pos)["predicted"]}
pd.DataFrame({k: compute_metrics(observed, v) for k, v in candidates.items()}).T.round(4)

AR(1) : phi = 0.263, écart-type des innovations = 1.100 kWh/kWc, BIC = 18977


,MSE,RMSE,MAE,MAPE,WAPE
persistance,2.0550,1.4335,0.9841,33.4744,21.9582
climatologie,1.2105,1.1002,0.8558,30.1909,19.0956
arima_fourier,1.1846,1.0884,0.8387,29.7417,18.7131


In [8]:
# Tous les modèles classiques, sur toutes les villes du mode choisi (reprise : une ville déjà traitée n'est pas refaite).
from solarvision.models import train as tr

results = tr.train_all(sorted(splits), fast=FAST, workers=3)
pd.DataFrame(results)[["ville", "statut"]]

17:02:06 madrid       déjà fait 


17:02:06 paris        déjà fait 


17:02:06 tunis        déjà fait 


,ville,statut
0,madrid,déjà fait
1,paris,déjà fait
2,tunis,déjà fait


## 5. Réseaux récurrents

LSTM et GRU par ville, en scénario ex ante puis ex post, et un LSTM global entraîné sur toutes les villes.
La cible est l'écart à la normale saisonnière. En mode rapide, l'entraînement est limité à trois époques :
il vérifie la chaîne, pas la performance.

In [9]:
from solarvision.models import deep as dp

for city in sorted(splits):
    print(dp.run_city(city, fast=FAST))
print(dp.run_global(sorted(splits), fast=FAST))
curves = pd.read_csv(dp.collect_curves(FAST))
curves[curves["ville"] == "tunis"].pivot_table(index="epoque", columns="modele", values="perte_validation").round(4)

{'ville': 'madrid', 'statut': 'déjà fait'}
{'ville': 'paris', 'statut': 'déjà fait'}
{'ville': 'tunis', 'statut': 'déjà fait'}
{'statut': 'déjà fait'}


modele,gru,gru_expost,lstm,lstm_expost
epoque,,,,
1,1.0008,0.9479,1.0112,0.8281
2,0.9947,0.7576,0.9993,0.6841
3,0.9931,0.6543,0.9935,0.5939


## 6. Évaluation

Mêmes origines, mêmes horizons, mêmes métriques pour tous les modèles. Les modèles **ex post** connaissent la météo
observée des jours visés : c'est une borne optimiste, jamais une performance opérationnelle.

In [10]:
from solarvision.evaluation import evaluate as ev

tables = ev.run(reports)
wide = tables["metrics_wide"]
view = wide[(wide["horizon"] == "tous") & (wide["technologie"] == "crystSi")]
view.pivot_table(index=["scenario", "modele"], columns="ville", values="RMSE").round(4)

ville                    madrid   paris   tunis
scenario modele                                
ex_ante  arima_fourier   1.2732  1.4136  1.0884
         climatologie    1.3170  1.4461  1.1002
         gru             1.2575  1.4052  1.0883
         lstm            1.2547  1.4081  1.0865
         lstm_global     1.2591  1.4024  1.0701
         persistance     1.5406  1.8735  1.4335
         prophet         1.3182  1.4439  1.1009
         xgboost         1.2681  1.4052  1.0861
ex_post  gru_expost      0.9699  1.1084  0.8805
         lstm_expost     0.9333  1.0648  0.8384
         xgboost_expost  0.1368  0.1800  0.1233

In [11]:
# Diebold-Mariano contre la climatologie (correction HLN, variance sur h - 1 retards, correction de Holm par famille).
dm = tables["dm_tests"]
dm[(dm["ville"] == "tunis") & (dm["technologie"] == "crystSi") & (dm["perte"] == "squared") & dm["horizon"].isin([1, 7])][
    ["modele", "scenario", "horizon", "dm_stat", "p_unilaterale", "p_holm_unilaterale", "taille_famille", "meilleur_que_reference"]].round(4)

,modele,scenario,horizon,dm_stat,p_unilaterale,p_holm_unilaterale,taille_famille,meilleur_que_reference
448,arima_fourier,ex_ante,1,5.3063,0.0000,0.0000,10,True
449,gru,ex_ante,1,6.1374,0.0000,0.0000,10,True
450,gru_expost,ex_post,1,8.6788,0.0000,0.0000,10,True
451,lstm,ex_ante,1,6.6160,0.0000,0.0000,10,True
452,lstm_expost,ex_post,1,9.5141,0.0000,0.0000,10,True
453,lstm_global,ex_ante,1,6.0861,0.0000,0.0000,10,True
454,persistance,ex_ante,1,-2.8658,0.9979,1.0000,10,False
455,prophet,ex_ante,1,-0.1103,0.5439,1.0000,10,False
456,xgboost,ex_ante,1,5.5017,0.0000,0.0000,10,True
457,xgboost_expost,ex_post,1,15.8994,0.0000,0.0000,10,True


In [12]:
tables["synthese_ville"][["ville", "meilleur_modele_ex_ante", "rmse_meilleur", "rmse_climatologie",
                           "gain_vs_climatologie_pct", "gain_h1_pct", "gain_h7_pct"]].round(3)

,ville,meilleur_modele_ex_ante,rmse_meilleur,rmse_climatologie,gain_vs_climatologie_pct,gain_h1_pct,gain_h7_pct
0,madrid,lstm,1.255,1.317,4.732,14.053,0.666
1,paris,lstm_global,1.402,1.446,3.019,11.352,0.201
2,tunis,lstm_global,1.070,1.100,2.734,10.778,0.577


## 7. Technologies et économie

Le score de fiabilité est calculé à partir de cinq critères ; quand l'écart entre deux technologies tient dans la marge
d'incertitude, elles sont déclarées ex æquo. Les hypothèses économiques sont **à vérifier** (`TODO_VERIFY`).

In [13]:
from solarvision.recommender import reliability

reco = reliability.compute(panel, wide, ev.load_forecasts(reports), reports)
reco["recommendation"][["ville", "technologie_recommandee", "score", "deuxieme_choix", "ecart_score",
                        "probabilite_meilleure", "ex_aequo"]].round(3)

,ville,technologie_recommandee,score,deuxieme_choix,ecart_score,probabilite_meilleure,ex_aequo
0,madrid,CIS,0.699,crystSi,0.046,0.892,True
1,paris,crystSi,0.779,CIS,0.079,0.940,True
2,tunis,crystSi,0.709,CIS,0.084,0.905,True


In [14]:
from solarvision.economics import simulate

eco = simulate.run(panel, reports, wide)
eco["ranking"][["ville", "meilleure_technologie_van", "van_p10", "van_p50", "van_p90", "temps_retour_p50",
                "ecart_non_concluant"]].round(1)

,ville,meilleure_technologie_van,van_p10,van_p50,van_p90,temps_retour_p50,ecart_non_concluant
0,madrid,crystSi,2090.5,4513.5,7880.7,8.4,True
2,tunis,crystSi,384.9,2022.7,4041.9,10.6,True
1,paris,crystSi,-800.2,1478.6,4194.4,13.2,True


## 8. Registre et suivi

Un modèle n'est promu que s'il bat la climatologie **et** la version en place, sur les mêmes dates de test.
En mode rapide, le registre est séparé (`models/registry_fast`) : il ne remplace jamais le modèle de l'étude complète.

In [15]:
from solarvision.config import MODELS
from solarvision.mlops import candidates, registry
from solarvision.monitoring import drift

if FAST:
    registry.use(MODELS / "registry_fast")
for r in candidates.register_and_promote(FAST):
    print(r["nom"], r.get("version"), "->", r["statut"])
active = registry.current()
summary = drift.run(panel, splits, ev.load_forecasts(reports), active["nom"] if active else "arima_fourier", reports)
print("\nDérive :", summary["derive"], "| jours en alerte d'erreur :", round(summary["part_des_jours_en_alerte_erreur_pct"], 1), "%")

2026/10/11 17:02:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas


2026/10/11 17:02:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables


2026/10/11 17:02:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types


2026/10/11 17:02:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints


2026/10/11 17:02:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults


2026/10/11 17:02:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments


2026/10/11 17:02:38 INFO alembic.runtime.migration: Context impl SQLiteImpl.


2026/10/11 17:02:38 INFO alembic.runtime.migration: Will assume non-transactional DDL.


lstm_global 1.1.0 -> promotion refusée : lstm_global 1.1.0 : ne bat pas la version en production lstm_global 1.0.0 (1.2439 contre 1.2439)
arima_fourier 1.1.0 -> promotion refusée : arima_fourier 1.1.0 : ne bat pas la version en production lstm_global 1.0.0 (1.2584 contre 1.2439)



Dérive : {'stable': 16, 'à surveiller': 4, 'alerte': 1} | jours en alerte d'erreur : 0.9 %


## 9. Pour aller plus loin

- Application et API : `python -m solarvision.cli serve --fast`, puis http://127.0.0.1:8001
- Rapport : `python -m solarvision.cli report --fast`
- Section IA générative : `python -m solarvision.cli llm-pack --fast`, puis les gabarits de `docs/llm/`
- Étude complète : `python -m solarvision.cli pipeline`